## Now to implement the Negative Binomial regression

In [1]:
import clean
import statsmodels.api as sm

In [2]:
sf_hazards, demographics, dictionary = clean.load_hazards_data(
    'calenviroscreen40resultsdatadictionary_F_2021.xlsx'
)
sf_hazards = clean.filter_san_francisco(sf_hazards)
print(f"SF census tracts: {len(sf_hazards)}")

SF census tracts: 195


In [3]:
sf_hazards, cleaned_covariates = clean.standardize(sf_hazards, clean.HAZARD_COVARIATES)

In [4]:
df = clean.prepare_analysis_data(
    ems_path='Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service_20260315.csv',
    hazards_path='calenviroscreen40resultsdatadictionary_F_2021.xlsx',
)

Standardized columns: ['ozone_std', 'pm2.5_std', 'diesel_pm_std', 'drinking_water_std', 'lead_std', 'pesticides_std', 'tox._release_std', 'traffic_std', 'cleanup_sites_std', 'groundwater_threats_std', 'haz._waste_std', 'imp._water_bodies_std', 'solid_waste_std', 'asthma_std', 'low_birth_weight_std', 'cardiovascular_disease_std', 'education_std', 'linguistic_isolation_std', 'poverty_std', 'unemployment_std', 'housing_burden_std']


In [5]:
df.head()

,census_tract,total_population,california_county,zip,approximate_location,longitude,latitude,ces_4.0_score,ces_4.0_percentile,ces_4.0_percentile_range,...,imp._water_bodies_std,solid_waste_std,asthma_std,low_birth_weight_std,cardiovascular_disease_std,education_std,linguistic_isolation_std,poverty_std,unemployment_std,housing_burden_std
0,6075023200,3972,San Francisco,94124,San Francisco,-122.386139,37.727755,54.605674,92.208775,90-95%,...,1.012849,6.286587,3.191068,1.946149,2.018325,1.306247,-0.097772,0.459864,0.761136,3.238608
1,6075023103,2890,San Francisco,94124,San Francisco,-122.375965,37.735632,50.508864,88.401412,85-90%,...,1.542051,4.083312,3.191068,1.731002,2.018325,0.891870,-0.644297,3.829840,0.880762,1.014756
2,6075023001,5398,San Francisco,94124,San Francisco,-122.401760,37.735192,49.329481,87.191125,85-90%,...,0.836449,1.461898,3.191068,0.913445,2.018325,1.518255,0.772963,0.717926,0.242757,1.513988
3,6075023400,3661,San Francisco,94124,San Francisco,-122.390088,37.721593,46.984298,84.404942,80-85%,...,1.012849,5.048250,3.191068,0.839680,2.018325,1.730262,1.078647,1.431389,0.840887,-0.785505
4,6075023102,4549,San Francisco,94124,San Francisco,-122.384498,37.734492,46.932707,84.329299,80-85%,...,1.542051,2.603741,3.191068,2.321119,2.018325,0.583496,-0.551666,2.038592,2.954277,1.498860


## In first iteration just evaluate one on column of average calls/week

In [7]:
std_covariates = []
for cov in clean.HAZARD_COVARIATES: 
    std_covariates.append(f'{cov}_std')

X = sm.add_constant(df[std_covariates])
y = df['call_avg_aug_oct']

model = sm.NegativeBinomial(y, X).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 3.399826
         Iterations: 24
         Function evaluations: 35
         Gradient evaluations: 35
                     NegativeBinomial Regression Results                      
Dep. Variable:       call_avg_aug_oct   No. Observations:                  182
Model:               NegativeBinomial   Df Residuals:                      160
Method:                           MLE   Df Model:                           21
Date:                Sat, 09 May 2026   Pseudo R-squ.:                  0.1754
Time:                        16:26:41   Log-Likelihood:                -618.77
converged:                       True   LL-Null:                       -750.39
Covariance Type:            nonrobust   LLR p-value:                 8.854e-44
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const 